# Module 2: Inference and Memory

In Module 1 you saw why owning the server pays off, but you still cannot see inside it. A request goes in, tokens come out, and the latency is one number you cannot break down. This module opens up a single request to the inference server underneath your agents. You watch it run in two phases, **prefill** then **decode**, measure where its time goes from the client, then read the same numbers from the server's own [vLLM](https://docs.vllm.ai) `/metrics`, and read the KV cache gauge that governs how many requests fit at once. Those three vocabulary words, TTFT, TPOT, and KV cache, are what every later module tunes against.

## Learning objectives
- Split one request into its two phases: prefill and decode
- Measure time to first token (TTFT) and time per output token (TPOT) from the client by streaming
- Connect prefill to TTFT and compute-bound work, decode to TPOT and bandwidth-bound work
- Read the server's own TTFT and TPOT from the vLLM `/metrics` endpoint
- Read the KV cache utilization gauge and say why it limits concurrency
- Name which phase dominates for a prompt-heavy versus an answer-heavy workload

## Prerequisites
- Finished Module 0 and Module 1, with a working endpoint and resolved settings
- A self-hosted vLLM endpoint in `VLLM_HOST`, set in Module 0
- The `/metrics` endpoint reachable, which `get_settings()` derives for you
- About 15 minutes

References: [Anatomy of vLLM](https://blog.vllm.ai/2025/09/05/anatomy-of-vllm.html) &middot; [vLLM production metrics](https://docs.vllm.ai/en/latest/design/metrics/) &middot; [PagedAttention paper](https://arxiv.org/abs/2309.06180) &middot; [Prometheus exposition format](https://prometheus.io/docs/instrumenting/exposition_formats/)

## Prefill and decode design basics

One request to an LLM runs in two phases, and they behave nothing alike.

- **Prefill** reads your whole prompt at once and builds the key/value cache for it. It is compute-bound, so the GPU is busy with large matrix multiplies. Prefill is what you wait through before the first token appears, so it sets **time to first token (TTFT)**.
- **Decode** generates the answer one token per step, and each step reads the growing KV cache out of GPU memory and appends one entry. It is bandwidth-bound, so the limit is how fast the GPU moves the cache, not raw compute. Decode sets the **time per output token (TPOT)**, also called inter-token latency.

Which phase dominates depends on your traffic. A long prompt with a short answer is prefill-heavy. A short prompt with a long answer is decode-heavy. Knowing which one your workload looks like tells you what to tune.

![One request runs in two phases: prefill processes the whole prompt and sets TTFT, then decode emits one token per step and sets TPOT, while the KV cache fills in prefill and grows one entry per decoded token](images/02_inference_and_memory_architecture.png)

## 1. Setup

This workshop reads its connection details from `common/config.py`, and parses the server's metrics with `common/metrics.py`. Install the dependencies this module needs. We reinstall here so this notebook stands on its own.

In [ ]:
%pip install -q "openai>=1.40" "requests>=2.31"

## 2. Configure endpoint and model

`get_settings()` reads `VLLM_HOST` and `MODEL_NAME` from your environment (Module 0 set them), and derives `metrics_url` from the same host so the two stay in sync. `build_client()` returns an OpenAI client pointed at your vLLM. You will use the client to send a request and the metrics URL to read what the server saw.

In [ ]:
# Setup: make common/ importable, resolve settings, build the client.
import os, sys, time
sys.path.insert(0, os.path.abspath(".."))

from common.config import get_settings, build_client
from common import metrics

settings = get_settings()
client = build_client(settings)
print("endpoint:", settings.vllm_host)
print("model   :", settings.model_name)
print("metrics :", settings.metrics_url)

**What you should see:** your endpoint, your model, and the derived metrics URL. All three come from your environment, so a missing variable fails loudly instead of pointing at someone else's server.

## 3. One plain request hides the split

Start the way most code calls a model: send a request, wait for the whole answer, and time it. This is the latency your users feel when you do not stream. It includes prefill plus all of decode, rolled into one number. Watch what it cannot tell you.

In [ ]:
# Requires a live vLLM endpoint.
# Send one non-streamed request and time the full round trip.
prompt = "List three reasons to run your own inference server. Keep each to one line."

start = time.time()
resp = client.chat.completions.create(
    model=settings.model_name,
    messages=[{"role": "user", "content": prompt}],
    max_tokens=128,
    temperature=0.0,
)
total = time.time() - start

print(resp.choices[0].message.content)
print(f"\ntotal latency: {total:.2f}s")
print("tokens:", resp.usage.completion_tokens, "completion /", resp.usage.prompt_tokens, "prompt")

**What you should see:** the full answer, one total latency, and token counts. That single number is the problem. It hides the split between prefill and decode, so you cannot tell whether you are waiting on the prompt or on the answer. To see the split, you have to stream.

## 4. Stream the same request and split the time

Streaming exposes the two phases. The first chunk arrives only after prefill finishes, so the gap from request to first chunk is your **TTFT**. Every chunk after that is one decode step, so the average gap between chunks is your **TPOT**. Measuring them apart is the only way to know which phase owns your latency.

In [ ]:
# Requires a live vLLM endpoint.
# Stream the response. First token gap = TTFT. Gaps between later tokens = TPOT.
start = time.time()
first_token_at = None
token_times = []

stream = client.chat.completions.create(
    model=settings.model_name,
    messages=[{"role": "user", "content": prompt}],
    max_tokens=128,
    temperature=0.0,
    stream=True,
)

text = []
for chunk in stream:
    delta = chunk.choices[0].delta.content
    if not delta:
        continue
    now = time.time()
    if first_token_at is None:
        first_token_at = now
    token_times.append(now)
    text.append(delta)

print("".join(text))

ttft = first_token_at - start
# Average gap between successive streamed chunks is the measured inter-token latency.
gaps = [b - a for a, b in zip(token_times, token_times[1:])]
tpot = sum(gaps) / len(gaps) if gaps else float("nan")

print(f"\nclient-measured TTFT: {ttft*1000:.0f} ms")
print(f"client-measured TPOT: {tpot*1000:.1f} ms/token  ({1/tpot:.1f} tokens/s)" if gaps else "")

**What you should see:** the streamed answer, a TTFT in the tens to low hundreds of milliseconds, and a TPOT of a few to tens of milliseconds per token. TTFT is your prefill cost. TPOT times the number of output tokens is your decode cost. For this short prompt and longer answer, decode dominates the total. That is the split the plain request in section 3 could never show you.

## 5. Read the same numbers from the server

Your client measured TTFT and TPOT from the outside, network hop included. vLLM measures them on the inside and exposes them as Prometheus histograms. You do not need a Prometheus server to read them: scraping `/metrics` once gives you the raw text, and the helper in `common/metrics.py` does the fetch and the parse. A histogram reports `_sum` and `_count`, so the running average is simply `sum / count`.

In [ ]:
# Requires a live vLLM endpoint.
# Pull the server-side TTFT and TPOT histogram averages from /metrics.
text = metrics.fetch_metrics_text(settings.metrics_url)

server_ttft = metrics.parse_histogram_avg(text, "vllm:time_to_first_token_seconds")
server_tpot = metrics.parse_histogram_avg(text, "vllm:time_per_output_token_seconds")

print(f"server avg TTFT: {server_ttft*1000:.0f} ms" if server_ttft else "TTFT: no samples yet")
print(f"server avg TPOT: {server_tpot*1000:.1f} ms/token" if server_tpot else "TPOT: no samples yet")

**What you should see:** server-side averages close to your client numbers, usually a little lower because they exclude the network hop. These are averages since the server started, so they smooth over the single request you just sent. Under load (Module 4) these same histograms are how you watch latency move.

## 6. Where the memory goes: the KV cache

During decode, every token attends to every previous token. To avoid recomputing, vLLM stores the keys and values for past tokens in GPU memory. That is the **KV cache**, and it is the scarce resource that decides how many requests you can run at once.

vLLM manages it with **PagedAttention**: instead of one big contiguous buffer per request, it splits the cache into fixed blocks of 16 tokens and hands them out on demand, like virtual memory pages. That is why it can pack many requests into one GPU without wasting memory on padding, and why the cache utilization gauge is the number you watch under load.

`kv_cache_usage_perc` is the fraction of those blocks in use, from 0 to 1. Idle it sits near zero. Push concurrency up (Modules 3 and 4) and watch it climb toward 1.0, where requests start queueing and getting preempted.

In [ ]:
# Requires a live vLLM endpoint.
# Read the live gauges: running, waiting, and KV cache utilization.
snap = metrics.snapshot(settings.metrics_url)

print(f"requests running : {snap['vllm:num_requests_running']:.0f}")
print(f"requests waiting : {snap['vllm:num_requests_waiting']:.0f}")
print(f"KV cache usage   : {snap['vllm:gpu_cache_usage_perc']*100:.1f}%")
print(f"preemptions total: {snap['vllm:num_preemptions_total']:.0f}")

**What you should see:** with only your own request in flight, `running` is 0 or 1, `waiting` is 0, KV cache usage is a few percent or less, and preemptions is 0. This is your idle baseline. Every later module pushes these numbers and asks you to read them.

## Things to know

- **TTFT is prefill, TPOT is decode.** One number for the whole request hides the split. Stream, and the first-chunk gap is prefill, the between-chunk gaps are decode. Tune the phase that dominates your traffic.
- **The gauge was renamed.** vLLM's V1 engine renamed the cache gauge from `vllm:gpu_cache_usage_perc` to `vllm:kv_cache_usage_perc`. The `snapshot()` helper reads whichever your server exposes and returns it under both names, so the code above works either way. In prose this workshop calls it `kv_cache_usage_perc`.
- **Client time includes the network, server time does not.** Expect the `/metrics` averages to run a little under your client numbers. Trust the server view for tuning, because that is the latency your server controls.
- **The cache is the concurrency ceiling.** PagedAttention blocks are finite. When `kv_cache_usage_perc` nears 1.0, vLLM has no free blocks, so new requests wait and running ones can be preempted. That single gauge explains most bottlenecks ahead.

> NOTE: A first request can show `no samples yet` from `/metrics` if the histogram has not recorded a completion. Run the streaming cell once, then re-read the metrics.

## Try it yourself

**Make prefill dominate.** Paste a long prompt (a few paragraphs) and ask for a one-line answer. Watch TTFT climb while TPOT and the token count stay small. This is the prefill-heavy shape from the design-basics figure. **Stretch:** double the prompt length and see TTFT roughly track it.

**Make decode dominate.** Keep the prompt short and raise `max_tokens` so the model writes a long answer. Watch the total grow with output tokens while TTFT barely moves. This is the decode-heavy shape, and it is bandwidth-bound.

**Watch the cache move.** Re-read `metrics.snapshot()` right after a long generation and compare `kv_cache_usage_perc` against the idle baseline. A single request barely moves it, which is exactly why concurrency, not one request, is what fills the cache.

In [ ]:
# Change PROMPT and MAX_TOKENS, then run the cell to shift the prefill/decode balance.
PROMPT = "Write one sentence on why you run your own inference."   # try a long prompt for prefill-heavy
MAX_TOKENS = 256                                                   # raise for decode-heavy

start = time.time()
first_token_at = None
token_times = []
stream = client.chat.completions.create(
    model=settings.model_name,
    messages=[{"role": "user", "content": PROMPT}],
    max_tokens=MAX_TOKENS,
    temperature=0.0,
    stream=True,
)
n = 0
for chunk in stream:
    if not chunk.choices[0].delta.content:
        continue
    now = time.time()
    if first_token_at is None:
        first_token_at = now
    token_times.append(now)
    n += 1

ttft = first_token_at - start
gaps = [b - a for a, b in zip(token_times, token_times[1:])]
tpot = sum(gaps) / len(gaps) if gaps else float("nan")
print(f"output tokens: {n}")
print(f"TTFT (prefill): {ttft*1000:.0f} ms")
print(f"TPOT (decode) : {tpot*1000:.1f} ms/token" if gaps else "TPOT: too few tokens")
print(f"KV cache usage: {metrics.snapshot(settings.metrics_url)['vllm:gpu_cache_usage_perc']*100:.1f}%")

## Summary

- One request runs in two phases. Prefill processes the whole prompt and is compute-bound. Decode emits one token per step and is bandwidth-bound.
- TTFT is your prefill cost, TPOT is your decode cost. Streaming is what lets you measure them apart instead of seeing one rolled-up latency.
- The server measures the same TTFT and TPOT itself and exposes them on `/metrics`, which is the view you tune against because it excludes the network.
- The KV cache, managed in 16-token PagedAttention blocks, is the scarce resource. `kv_cache_usage_perc` is the gauge that tells you how close you are to the concurrency ceiling.

## Next

**Module 3: Serving with vLLM.** You measured one request in isolation. Next you fire several at once and watch continuous batching pack them into a single running batch, raising throughput without a proportional hit to the TTFT and TPOT you just learned to read.